# 02 Exploratory data analysis

**Goal.** Look at the loan leads before modelling anything: who applies, how much they borrow,
what they earn and owe, when leads arrive, and how often a lead converts
(`deducted_within_30_days`, the target). The same notebook can **compare two datasets**, so
the synthetic stand-in can later be checked against the company's real export.

**How it is meant to be used.**

- Set `DATA_PATH` (and optionally `COMPARE_PATH`) in the parameter cell below. Both files must have
  exactly the columns of [`docs/data/data-dictionary.md`](../docs/data/data-dictionary.md).
- With `COMPARE_PATH = None` the notebook explores one dataset. With a second path, every plot shows both
  datasets and the last section scores each column with a verdict (similar / moderate shift / shifted).
- The notebook shows aggregates only (counts, shares, statistics, plots), never individual rows. Outputs are not
  committed (nbstripout). If a run on production data is exported (HTML/PDF), review it before it goes to
  `docs/presentations/` or anywhere else outside this machine.

**Units** (as assumed by the generator, to be confirmed on real data): money in EUR per month, durations in
months (shown here in years), age in years, flags 0/1, `employment_duration` = 0 means not currently employed.

**Limitation.** Right now the data is synthetic. Every relationship shown here is an assumption of the
generator, so these results say nothing about how real leads behave. They show that the analysis works and what
to look for once real data arrives.

## Parameters

In [ ]:
# This cell is tagged "parameters", so the notebook can be driven by papermill later.
DATA_PATH = "../data/synthetic_leads.csv"
COMPARE_PATH = None      # e.g. "../data/production_leads.csv"; None = explore DATA_PATH alone
DATA_LABEL = None        # name in legends and tables; None = file name without extension
COMPARE_LABEL = None

SIMILAR_BELOW = 0.10     # column score below this: "similar"
SHIFTED_FROM = 0.20      # column score from this: "shifted" (in between: "moderate shift")

DICTIONARY = "../docs/data/data-dictionary.md"

## Load and validate

Column groups are listed explicitly (the dictionary has no types yet). The notebook stops if they do not cover
exactly the dictionary's fields, or if a dataset's columns differ from the dictionary.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import ks_2samp

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
COLORS = ["#4C78A8", "#F58518"]

TARGET = "deducted_within_30_days"
ID_COLS = ["application_id", "prediction_at"]
NUMERIC = [
    "loan_amount", "loan_duration", "age", "main_income", "other_income", "total_expenses",
    "disposable_income", "open_credit_count", "open_credit_monthly_installments",
    "open_credit_residual_debt", "number_of_children", "employment_duration", "residence_duration",
]
CATEGORICAL = [
    "loan_purpose", "relationship_between_applicants", "living_arrangements", "occupation_group", "education",
]
FLAGS = ["two_applicants", "temporary_job", "mini_job", "in_probation", TARGET]
SCORED = NUMERIC + CATEGORICAL + FLAGS   # ids and time are not compared


def dictionary_fields(path):
    return re.findall(r"^\| `(\w+)` \|", Path(path).read_text(), flags=re.M)


FIELDS = dictionary_fields(DICTIONARY)
GROUPED = ID_COLS + SCORED
assert sorted(GROUPED) == sorted(FIELDS), (
    f"Column groups differ from the data dictionary. Not grouped: {sorted(set(FIELDS) - set(GROUPED))}. "
    f"Not in dictionary: {sorted(set(GROUPED) - set(FIELDS))}."
)


def load_dataset(path):
    df = pd.read_csv(path)
    missing = [c for c in FIELDS if c not in df.columns]
    extra = [c for c in df.columns if c not in FIELDS]
    if missing or extra:
        raise ValueError(
            f"{path}: columns do not match the data dictionary. Missing: {missing}. Extra: {extra}."
        )
    df = df[FIELDS].copy()
    df["prediction_at"] = pd.to_datetime(df["prediction_at"])
    return df


labels = [DATA_LABEL or Path(DATA_PATH).stem]
if COMPARE_PATH:
    labels.append(COMPARE_LABEL or Path(COMPARE_PATH).stem)
    if labels[1] == labels[0]:
        labels[1] += " (compare)"

data = {labels[0]: load_dataset(DATA_PATH)}
if COMPARE_PATH:
    data[labels[1]] = load_dataset(COMPARE_PATH)
COMPARING = len(data) == 2

for label, d in data.items():
    print(f"{label}: {len(d):,} rows, {d.shape[1]} columns")
print("Mode:", "comparison of two datasets" if COMPARING else "single dataset")

In [ ]:
def prepare(df):
    """Display-only helper columns; nothing here is saved or used as a model feature."""
    v = df.copy()
    v["employment_years"] = (v.employment_duration / 12).where(v.employment_duration > 0)  # employed only
    v["residence_years"] = v.residence_duration / 12
    v["age_group"] = pd.cut(
        v.age, [17, 24, 34, 44, 54, 64, 150], labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
    )
    v["disposable_income_group"] = pd.cut(
        v.disposable_income, [-np.inf, 0, 500, 1000, 2000, 4000, np.inf],
        labels=["<= 0", "0-500", "500-1k", "1k-2k", "2k-4k", "4k+"],
    )
    # Approximate monthly payment of the new loan: amount / months, interest ignored.
    v["payment_approx"] = v.loan_amount / v.loan_duration
    ratio = (v.payment_approx / v.disposable_income).where(v.disposable_income > 0)
    group = pd.cut(
        ratio, [0, 0.1, 0.25, 0.5, 1, np.inf], labels=["< 10%", "10-25%", "25-50%", "50-100%", "> 100%"]
    ).astype(object)
    group[(v.disposable_income <= 0).to_numpy()] = "no disposable income"
    v["payment_to_disposable"] = pd.Categorical(
        group, categories=["no disposable income", "< 10%", "10-25%", "25-50%", "50-100%", "> 100%"], ordered=True
    )
    return v


view = {label: prepare(d) for label, d in data.items()}


def plot_numeric(cols, ncols=3, clip=True, max_discrete=15):
    """Overlay distributions as densities (or shares for columns with few distinct values)."""
    nrows = -(-len(cols) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 3.2 * nrows), squeeze=False)
    flat = axes.ravel()
    for ax, col in zip(flat, cols):
        pooled = pd.concat([d[col] for d in view.values()]).dropna()
        if pooled.nunique() <= max_discrete:
            values = np.sort(pooled.unique())
            width = 0.8 / len(view)
            for i, (label, d) in enumerate(view.items()):
                s = d[col].value_counts(normalize=True).reindex(values, fill_value=0)
                ax.bar(np.arange(len(values)) + (i - (len(view) - 1) / 2) * width, s.values, width,
                       color=COLORS[i], label=label)
            ax.set_xticks(range(len(values)))
            ax.set_xticklabels([f"{v:g}" for v in values])
            ax.set_ylabel("share")
        else:
            lo, hi = pooled.quantile([0.01, 0.99]) if clip else (pooled.min(), pooled.max())
            bins = np.linspace(lo, hi, 41)
            for i, (label, d) in enumerate(view.items()):
                x = d[col].dropna()
                ax.hist(x[(x >= lo) & (x <= hi)], bins=bins, density=True, alpha=0.55, color=COLORS[i], label=label)
            ax.set_ylabel("density")
            if clip:
                ax.set_xlabel("1st to 99th percentile shown")
        ax.set_title(col)
    for ax in flat[len(cols):]:
        ax.axis("off")
    flat[0].legend()
    fig.tight_layout()
    plt.show()


def plot_categorical(cols, ncols=3):
    """Side-by-side category shares (missing values left out; see the missing-value table)."""
    nrows = -(-len(cols) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 3.4 * nrows), squeeze=False)
    flat = axes.ravel()
    for ax, col in zip(flat, cols):
        pooled = pd.concat([d[col] for d in view.values()]).dropna()
        order = list(pooled.cat.categories) if isinstance(pooled.dtype, pd.CategoricalDtype) \
            else list(pooled.value_counts().index)
        width = 0.8 / len(view)
        for i, (label, d) in enumerate(view.items()):
            s = d[col].dropna().value_counts(normalize=True).reindex(order, fill_value=0)
            ax.bar(np.arange(len(order)) + (i - (len(view) - 1) / 2) * width, s.values, width,
                   color=COLORS[i], label=label)
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels([str(o) for o in order], rotation=35, ha="right")
        ax.set_ylabel("share")
        ax.set_title(col)
    for ax in flat[len(cols):]:
        ax.axis("off")
    flat[0].legend()
    fig.tight_layout()
    plt.show()


def conversion_by(by):
    """Conversion rate (%) and row count per group, one column per dataset."""
    parts = {label: d.groupby(by, observed=True)[TARGET].agg(["mean", "size"]) for label, d in view.items()}
    rate = pd.DataFrame({l: p["mean"] * 100 for l, p in parts.items()})
    n = pd.DataFrame({l: p["size"] for l, p in parts.items()})
    return rate, n


def plot_rates(tables, ncols=2):
    """tables: {title: rate DataFrame}; bars side by side per dataset."""
    nrows = -(-len(tables) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.2 * ncols, 3.4 * nrows), squeeze=False)
    flat = axes.ravel()
    for ax, (title, rate) in zip(flat, tables.items()):
        width = 0.8 / rate.shape[1]
        for i, label in enumerate(rate.columns):
            ax.bar(np.arange(len(rate)) + (i - (rate.shape[1] - 1) / 2) * width, rate[label].values, width,
                   color=COLORS[i], label=label)
        ax.set_xticks(range(len(rate)))
        ax.set_xticklabels([str(i) for i in rate.index], rotation=35, ha="right")
        ax.set_ylabel("conversion rate (%)")
        ax.set_title(title)
    for ax in flat[len(tables):]:
        ax.axis("off")
    flat[0].legend()
    fig.tight_layout()
    plt.show()

## 1. Overview

Types, summary statistics and the share of missing values per column. Missing values are intended in some
columns (education, residence and employment duration, children, occupation) and structural in
`relationship_between_applicants`, which exists only for two applicants.

In [ ]:
types = pd.DataFrame({"dtype": next(iter(data.values())).dtypes.astype(str)})
display(types)

missing = pd.DataFrame({label: d.isna().mean() * 100 for label, d in data.items()}).round(1)
print("Missing values per column (%), columns with gaps only:")
display(missing[(missing > 0).any(axis=1)])

for label, d in data.items():
    print(f"\nSummary statistics: {label}")
    display(d[NUMERIC].describe().T.round(1))

## 2. Applicants

Age, household and background. Missing categories are left out of these plots.

In [ ]:
plot_numeric(["age", "number_of_children"], ncols=2)
plot_categorical(["two_applicants", "relationship_between_applicants", "living_arrangements"])
plot_categorical(["occupation_group", "education"], ncols=2)

## 3. Employment and residence

Durations are stored in months and shown here in years (months / 12). `employment_duration = 0` means not
currently employed; those leads are reported separately and left out of the tenure plot.

In [ ]:
rows = {}
for label, d in data.items():
    emp = d.employment_duration.dropna()
    rows[label] = {
        "not employed (%)": (emp == 0).mean() * 100,
        "in probation (%)": d.in_probation.mean() * 100,
        "mini job (%)": d.mini_job.mean() * 100,
        "temporary job (%)": d.temporary_job.mean() * 100,
    }
display(pd.DataFrame(rows).round(1))

plot_numeric(["employment_years", "residence_years"], ncols=2)

## 4. Income and existing credit

Money amounts in EUR per month, except residual debt (EUR total). Plots are cut at the 1st and 99th percentile
for readability; the comparison scores later use all values.

In [ ]:
rows = {}
for label, d in data.items():
    rows[label] = {
        "no disposable income, <= 0 (%)": (d.disposable_income <= 0).mean() * 100,
        "any open credit (%)": (d.open_credit_count > 0).mean() * 100,
        "no other income (%)": (d.other_income == 0).mean() * 100,
    }
display(pd.DataFrame(rows).round(1))

plot_numeric(["main_income", "other_income", "total_expenses", "disposable_income"], ncols=2)
plot_numeric(["open_credit_count", "open_credit_monthly_installments", "open_credit_residual_debt"])

## 5. The requested loan

How much, for how long and for what.

In [ ]:
plot_numeric(["loan_amount", "loan_duration"], ncols=2)
plot_categorical(["loan_purpose"], ncols=1)

amount = pd.DataFrame({label: d.groupby("loan_purpose").loan_amount.median() for label, d in data.items()})
print("Median loan amount by purpose (EUR):")
display(amount.round(0))

## 6. Lead timing

Lead volume by hour, weekday and month, one row per dataset (not overlaid, since two datasets rarely cover the
same period).

In [ ]:
fig, axes = plt.subplots(len(view), 3, figsize=(14, 3.0 * len(view)), squeeze=False)
for r, (label, d) in enumerate(data.items()):
    t = d.prediction_at
    parts = [(t.dt.hour, "hour of day"), (t.dt.day_of_week, "weekday (0 = Monday)"), (t.dt.month, "month")]
    for c, (part, name) in enumerate(parts):
        s = part.value_counts(normalize=True).sort_index()
        axes[r, c].bar(s.index, s.values, color=COLORS[r])
        axes[r, c].set_title(f"{label}: share of leads by {name}", fontsize=9)
        axes[r, c].set_ylabel("share")
    print(f"{label}: {t.min():%Y-%m-%d} to {t.max():%Y-%m-%d}")
fig.tight_layout()
plt.show()

## 7. Conversion (the target)

`deducted_within_30_days` is 1 when the loan was paid out within 30 days of the lead. It is rare, so a model that
always predicts 0 is already right most of the time; accuracy will not be a useful metric.

In [ ]:
for label, d in data.items():
    rate = d[TARGET].mean()
    print(f"{label}: conversion rate {rate:.2%} ({int(d[TARGET].sum()):,} of {len(d):,}); "
          f"always predicting 0 is right {1 - rate:.1%} of the time")

In [ ]:
tables = {}
counts = {}
for title, by in [("loan purpose", "loan_purpose"), ("age group", "age_group"),
                  ("disposable income group (EUR/month)", "disposable_income_group")]:
    tables[title], counts[title] = conversion_by(by)

flag_rates, flag_counts = [], []
for flag in ["in_probation", "mini_job", "temporary_job", "two_applicants"]:
    r, n = conversion_by(flag)
    r.index = [f"{flag} = {i}" for i in r.index]
    n.index = r.index
    flag_rates.append(r)
    flag_counts.append(n)
tables["employment and applicant flags"] = pd.concat(flag_rates)
counts["employment and applicant flags"] = pd.concat(flag_counts)

plot_rates(tables)
for title, n in counts.items():
    print(f"Leads per group: {title}")
    display(n.T)

### Affordability

The approximate monthly payment (loan amount / duration, no interest) relative to what is left after expenses and
existing credit. For exploration only: this is not saved as a feature.

In [ ]:
rate, n = conversion_by("payment_to_disposable")
plot_rates({"payment / disposable income": rate}, ncols=1)
display(n.T)

## 8. Relationships

Rank correlation (Spearman) between the numeric columns and the target, per dataset.

In [ ]:
cols = NUMERIC + [TARGET]
fig, axes = plt.subplots(1, len(view), figsize=(7.5 * len(view), 6.5), squeeze=False)
for ax, (label, d) in zip(axes[0], data.items()):
    corr = d[cols].corr(method="spearman")
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels(cols, rotation=75, ha="right", fontsize=8)
    ax.set_yticks(range(len(cols)))
    ax.set_yticklabels(cols, fontsize=8)
    ax.set_title(label)
fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8)
plt.show()

## 9. Comparison of the two datasets

Only runs when `COMPARE_PATH` is set. Each column gets a distance between the two datasets, from 0 (same
distribution) to 1 (no overlap), computed on non-missing values:

- numeric columns: **Kolmogorov-Smirnov statistic**, the largest gap between the two cumulative distributions;
- categorical columns and 0/1 flags: **total variation distance**, half the sum of the absolute differences in
  category shares.

A verdict compares the distance with `SIMILAR_BELOW` and `SHIFTED_FROM`. The thresholds are rules of thumb, not
standards. A verdict flags a difference worth inspecting in the plots above; it does not say that either dataset is
wrong. `application_id` and `prediction_at` are not scored.

In [ ]:
def ks_distance(a, b):
    return ks_2samp(a, b).statistic


def tv_distance(a, b):
    pa = a.value_counts(normalize=True)
    pb = b.value_counts(normalize=True)
    idx = pa.index.union(pb.index)
    return 0.5 * (pa.reindex(idx, fill_value=0) - pb.reindex(idx, fill_value=0)).abs().sum()


def verdict(score):
    if np.isnan(score):
        return "not comparable"
    if score < SIMILAR_BELOW:
        return "similar"
    return "moderate shift" if score < SHIFTED_FROM else "shifted"


if not COMPARING:
    print("Comparison skipped: set COMPARE_PATH in the parameter cell to compare two datasets.")
else:
    first, second = data.values()
    rows = []
    for col in SCORED:
        a, b = first[col].dropna(), second[col].dropna()
        kind = "numeric" if col in NUMERIC else "categorical"
        if a.empty or b.empty:
            score = np.nan
        else:
            score = ks_distance(a, b) if kind == "numeric" else tv_distance(a, b)
        rows.append({"field": col, "kind": kind, "score": score, "verdict": verdict(score)})
    scores = pd.DataFrame(rows).sort_values("score", ascending=False, na_position="last").reset_index(drop=True)
    display(scores.round(3))

    shifted = scores.loc[scores.verdict == "shifted", "field"].tolist()
    moderate = scores.loc[scores.verdict == "moderate shift", "field"].tolist()
    unknown = scores.loc[scores.verdict == "not comparable", "field"].tolist()
    if shifted:
        print(f"{len(shifted)} of {len(scores)} fields shifted: {', '.join(shifted)}.")
    else:
        print(f"No field is shifted (score >= {SHIFTED_FROM}).")
    if moderate:
        print(f"Moderate shift: {', '.join(moderate)}.")
    if unknown:
        print(f"Not comparable (empty in one dataset): {', '.join(unknown)}.")

    plotted = scores.dropna(subset=["score"]).iloc[::-1]
    palette = {"similar": "#54A24B", "moderate shift": "#ECB000", "shifted": "#D03A3A"}
    fig, ax = plt.subplots(figsize=(7, 0.28 * len(plotted) + 1.2))
    ax.barh(plotted.field, plotted.score, color=plotted.verdict.map(palette))
    ax.axvline(SIMILAR_BELOW, color="grey", ls="--", lw=1)
    ax.axvline(SHIFTED_FROM, color="grey", ls="--", lw=1)
    ax.set_xlim(0, 1)
    ax.set_xlabel("distance between datasets (KS or total variation)")
    plt.show()

In [ ]:
if COMPARING:
    first, second = data.values()
    l1, l2 = list(data)

    rate1, rate2 = first[TARGET].mean(), second[TARGET].mean()
    print(f"Conversion rate: {l1} {rate1:.2%}, {l2} {rate2:.2%}; difference {100 * (rate2 - rate1):+.2f} points, "
          f"relative {(rate2 / rate1 - 1) if rate1 else float('nan'):+.0%}")

    by_purpose, _ = conversion_by("loan_purpose")
    by_purpose["difference (points)"] = by_purpose[l2] - by_purpose[l1]
    print("\nConversion rate by loan purpose (%):")
    display(by_purpose.round(2))

    print("Missing values per column (%), side by side:")
    display(missing[(missing > 0).any(axis=1)])

## Conclusion

**What the synthetic data looks like** (50,000 leads, 2025-01-01 to 2025-12-31). Applicants are 18 to 75 years old
(mean 41); 14.5% are not currently employed, the median tenure of the employed is about 4.6 years, 3.9% are in
probation and 5.5% have a mini job. Median main income is about 3,000 EUR per month and median disposable income
about 2,200 EUR; 11% have no disposable income and 55% have open credit. Loans are small and short: median
7,300 EUR and 36 months, with new cars (median 14,800 EUR) and debt conversion (10,600 EUR) at the top and
electronics (2,700 EUR) at the bottom. Leads arrive mostly during the day, evenly across weekdays and months.

**Conversion.** 3.0% of leads convert, so accuracy is not a useful metric (always predicting 0 is right 97% of the
time). Conversion is highest for `dispo` (4.2%) and `debt_conversion` (3.8%) and lowest for `journey` (1.9%) and
`other` (2.0%). It is close to zero without disposable income (0.3%) and when the payment exceeds the disposable
income (0.5%), low for probation (1.1%) and mini jobs (1.1%), and lower for the youngest (1.5%) and oldest
(2.0%) applicants than for ages 25 to 64 (2.9 to 3.3%).

**These are assumptions, not findings.** Every pattern above was put into the data by the generator
(`01_synthetic_data.ipynb`), so it says nothing about how real leads behave. The value of this notebook is that it
runs on any file with the dictionary's columns.

**Next.** When the real export arrives, set `DATA_PATH` to the synthetic file and `COMPARE_PATH` to the real one.
Look first at the fields marked "shifted" (unit mismatches show up as scores near 1), at the conversion rate, and at
the purpose shares; then update the generator's assumptions where they differ. The verdict thresholds are
defaults and may need adjusting once real data shows how large normal differences are. Comparison mode was
checked by comparing the file with itself (all scores 0), with a copy whose income is multiplied by 12
(main income 0.91, other income 0.67, shifted) and with a regeneration using another seed (all scores
at most 0.01, all similar).